### One-to-One (1:1)

Sometimes a row in table A relates to *exactly one* row in table B — for example, a `user` and their `user_settings`. You model this with a foreign key plus a `UNIQUE` constraint on it (or by making the FK the primary key):

```sql
CREATE TABLE user_settings (
    user_id INTEGER PRIMARY KEY REFERENCES users(id) ON DELETE CASCADE,
    theme   VARCHAR(20) DEFAULT 'light'
);
```

Why split a 1:1 instead of adding columns to `users`?

- Different access patterns (`users` is read constantly, `settings` rarely)
- Different security/permissions (e.g. PII separated)
- Optional data — a `NULL` row is cheaper than many `NULL` columns

We'll see this exact pattern again in Notebook 4 when we **split** a wide table.


# 📖 Notebook 1: Relational Modeling Basics

Before we explore advanced patterns, let's build a solid foundation: **entities, keys, relationships, and constraints**.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to identify entities and map them to tables
- Primary keys vs foreign keys and why they matter
- One-to-many (1:N) and many-to-many (N:M) relationships
- Constraints that keep your data correct
- How indexes speed up queries

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 01-foundations/data-modeling
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `data_modeling_demo`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [1]:
import psycopg2
import psycopg2.extras

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "data_modeling_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def query(sql, params=None):
    """Run a SELECT and return rows as dictionaries."""
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute(sql, params)
    rows = cursor.fetchall()
    conn.close()
    return rows

def execute(sql, params=None):
    """Run an INSERT/UPDATE/DELETE and commit."""
    conn = get_db()
    cursor = conn.cursor()
    cursor.execute(sql, params)
    conn.commit()
    conn.close()

# Test the connection
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

✅ Connected to PostgreSQL


## 🧱 What Are Entities?

An **entity** is a "thing" your application cares about. In a social media app, the entities are:

- **Users** — people on the platform
- **Posts** — content created by users
- **Comments** — replies to posts
- **Likes** — reactions to posts
- **Follows** — connections between users

Each entity becomes a **table** in a relational database. Each row is one instance (one specific user, one specific post).

Let's look at what's already in our database:

In [2]:
# List all tables in our database
tables = query("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")

print("📋 Tables in our database:")
print("=" * 40)
for t in tables:
    count = query(f"SELECT COUNT(*) as n FROM {t['table_name']}")
    print(f"  {t['table_name']:<25} ({count[0]['n']} rows)")

📋 Tables in our database:
  comments                  (500 rows)
  follows                   (522 rows)
  likes                     (1397 rows)
  post_tags                 (365 rows)
  posts                     (200 rows)
  posts_denormalized        (200 rows)
  tags                      (10 rows)
  users                     (50 rows)


## 🔑 Primary Keys

Every table needs a **primary key** — a column (or set of columns) that uniquely identifies each row.

**Rules:**
- Must be **unique** — no two rows can have the same value
- Must be **not null** — every row must have a value
- Should be **stable** — don't change after creation

**Best practice:** Use a system-generated ID (like `SERIAL` in Postgres) rather than business data like emails. Why? Emails can change. IDs don't.

```sql
CREATE TABLE users (
    id SERIAL PRIMARY KEY,  -- auto-incrementing integer
    username VARCHAR(50) UNIQUE NOT NULL,
    email VARCHAR(255) UNIQUE NOT NULL
);
```

In [3]:
# Look at some users — notice each has a unique id
users = query("SELECT id, username, email, display_name FROM users ORDER BY id LIMIT 5")

print("👤 Sample users:")
print(f"{'id':<5} {'username':<15} {'email':<30} {'display_name'}")
print("-" * 70)
for u in users:
    print(f"{u['id']:<5} {u['username']:<15} {u['email']:<30} {u['display_name']}")

print()
print("💡 The 'id' column is the primary key — unique, not null, auto-generated.")
print("   'username' and 'email' are also UNIQUE, but they're business data that could change.")

👤 Sample users:
id    username        email                          display_name
----------------------------------------------------------------------
1     user_1          user_1@example.com             User 1
2     user_2          user_2@example.com             User 2
3     user_3          user_3@example.com             User 3
4     user_4          user_4@example.com             User 4
5     user_5          user_5@example.com             User 5

💡 The 'id' column is the primary key — unique, not null, auto-generated.
   'username' and 'email' are also UNIQUE, but they're business data that could change.


## 🔗 Foreign Keys & Relationships

A **foreign key** is a column in one table that points to the primary key of another table. It creates a **relationship** between the two.

### One-to-Many (1:N)

One user can have **many** posts. Each post belongs to **one** user.

```
users (1) ──────< posts (N)
  id  ◄──────── user_id
```

The `posts.user_id` column is a foreign key that references `users.id`.

In [4]:
# One-to-Many: find all posts by user_1
posts = query("""
    SELECT p.id, p.content, p.created_at
    FROM posts p
    WHERE p.user_id = 1
    ORDER BY p.created_at DESC
    LIMIT 5;
""")

print("📝 Posts by user_1 (one-to-many relationship):")
print("=" * 70)
for p in posts:
    print(f"  Post #{p['id']}: {p['content'][:60]}...")
    print(f"           Created: {p['created_at']}")
    print()

print(f"💡 user_1 has {len(posts)} posts shown (limited to 5). Each post's user_id = 1.")

📝 Posts by user_1 (one-to-many relationship):
  Post #126: Just shipped a new feature at work! Feeling productive 🚀...
           Created: 2026-04-08 11:48:14.604843

  Post #66: Just shipped a new feature at work! Feeling productive 🚀...
           Created: 2026-03-26 09:19:30.348372

  Post #140: Made homemade pasta for the first time — turned out amazing ...
           Created: 2026-03-17 21:01:02.847085

  Post #35: Working on a side project this weekend. Building a URL short...
           Created: 2026-01-31 05:29:02.647941

  Post #56: Made homemade pasta for the first time — turned out amazing ...
           Created: 2026-01-29 05:00:07.508027

💡 user_1 has 5 posts shown (limited to 5). Each post's user_id = 1.


In [5]:
# JOINs let you combine data from related tables
# This is the power of relational databases!

results = query("""
    SELECT u.username, p.content, p.created_at
    FROM posts p
    JOIN users u ON p.user_id = u.id
    ORDER BY p.created_at DESC
    LIMIT 5;
""")

print("📰 Recent posts with usernames (using JOIN):")
print("=" * 70)
for r in results:
    print(f"  @{r['username']}: {r['content'][:55]}")
    print(f"  {'':>10} {r['created_at']}")
    print()

print("💡 The JOIN connects posts to users via the foreign key (user_id → id).")
print("   Without JOINs, you'd need two separate queries and stitch them in code.")

📰 Recent posts with usernames (using JOIN):
  @user_15: Made homemade pasta for the first time — turned out ama
             2026-04-18 13:51:02.410133

  @user_29: Working on a side project this weekend. Building a URL 
             2026-04-18 08:07:52.434605

  @user_29: Just shipped a new feature at work! Feeling productive 
             2026-04-18 03:05:24.458211

  @user_48: Just shipped a new feature at work! Feeling productive 
             2026-04-17 04:39:59.966798

  @user_4: Currently reading "Designing Data-Intensive Application
             2026-04-17 03:31:31.735215

💡 The JOIN connects posts to users via the foreign key (user_id → id).
   Without JOINs, you'd need two separate queries and stitch them in code.


### Many-to-Many (N:M)

Users can like **many** posts. Posts can be liked by **many** users. This is a **many-to-many** relationship.

In SQL, many-to-many relationships need a **junction table** (also called a join table or bridge table):

```
users (N) ──────< likes >──────  posts (M)
  id  ◄──────── user_id
                 post_id ────────► id
```

The `likes` table has two foreign keys — one pointing to `users`, one to `posts`.

In [6]:
# Many-to-Many: who liked post #1?
likers = query("""
    SELECT u.username, l.created_at
    FROM likes l
    JOIN users u ON l.user_id = u.id
    WHERE l.post_id = 1
    ORDER BY l.created_at;
""")

print(f"❤️  Users who liked post #1 ({len(likers)} total):")
for liker in likers[:10]:
    print(f"  @{liker['username']} — liked at {liker['created_at']}")
if len(likers) > 10:
    print(f"  ... and {len(likers) - 10} more")

print()
print("💡 The likes table is a junction table — it connects users to posts.")
print("   UNIQUE(user_id, post_id) prevents the same user from liking a post twice.")

❤️  Users who liked post #1 (4 total):
  @user_24 — liked at 2026-02-24 16:21:14.205861
  @user_1 — liked at 2026-02-26 23:57:41.448972
  @user_47 — liked at 2026-03-11 14:39:05.960057
  @user_50 — liked at 2026-04-18 14:10:31.257233

💡 The likes table is a junction table — it connects users to posts.
   UNIQUE(user_id, post_id) prevents the same user from liking a post twice.


In [7]:
# Self-referencing many-to-many: the follows table
# Both follower_id and following_id point to users.id

follows_data = query("""
    SELECT
        f1.username AS follower,
        f2.username AS following
    FROM follows fl
    JOIN users f1 ON fl.follower_id = f1.id
    JOIN users f2 ON fl.following_id = f2.id
    WHERE fl.follower_id = 1
    LIMIT 10;
""")

print("👥 Who does user_1 follow? (self-referencing N:M):")
print("=" * 40)
for f in follows_data:
    print(f"  @{f['follower']} follows → @{f['following']}")

print()
print("💡 The follows table references users TWICE (follower_id, following_id).")
print("   CHECK (follower_id != following_id) prevents users from following themselves.")

👥 Who does user_1 follow? (self-referencing N:M):
  @user_1 follows → @user_6
  @user_1 follows → @user_19
  @user_1 follows → @user_26
  @user_1 follows → @user_28
  @user_1 follows → @user_36
  @user_1 follows → @user_46
  @user_1 follows → @user_49

💡 The follows table references users TWICE (follower_id, following_id).
   CHECK (follower_id != following_id) prevents users from following themselves.


### One-to-One (1:1)

Sometimes a row in table A relates to *exactly one* row in table B — for example, a `user` and their `user_settings`. You model this with a foreign key plus a `UNIQUE` constraint on it (or by making the FK the primary key):

```sql
CREATE TABLE user_settings (
    user_id INTEGER PRIMARY KEY REFERENCES users(id) ON DELETE CASCADE,
    theme   VARCHAR(20) DEFAULT 'light'
);
```

Why split a 1:1 instead of adding columns to `users`?

- Different access patterns (`users` is read constantly, `settings` rarely)
- Different security/permissions (e.g. PII separated)
- Optional data — a `NULL` row is cheaper than many `NULL` columns

We'll see this exact pattern again in Notebook 4 when we **split** a wide table.


## 🛡️ Constraints

Constraints enforce **rules** at the database level so bad data can't sneak in:

| Constraint | What It Does | Example |
|------------|-------------|----------|
| `PRIMARY KEY` | Unique + Not Null identifier | `id SERIAL PRIMARY KEY` |
| `FOREIGN KEY` | Must reference existing row | `user_id REFERENCES users(id)` |
| `UNIQUE` | No duplicate values | `email VARCHAR(255) UNIQUE` |
| `NOT NULL` | Can't be empty | `username VARCHAR(50) NOT NULL` |
| `CHECK` | Custom rule | `CHECK (follower_id != following_id)` |

Let's see them in action:

In [8]:
# Constraint demo: try to violate each one and see what happens

import psycopg2.errors

demos = [
    (
        "UNIQUE constraint (duplicate email)",
        "INSERT INTO users (username, email) VALUES ('new_user', 'user_1@example.com')"
    ),
    (
        "FOREIGN KEY constraint (non-existent user)",
        "INSERT INTO posts (user_id, content) VALUES (99999, 'orphan post')"
    ),
    (
        "NOT NULL constraint (missing content)",
        "INSERT INTO posts (user_id, content) VALUES (1, NULL)"
    ),
    (
        "CHECK constraint (follow yourself)",
        "INSERT INTO follows (follower_id, following_id) VALUES (1, 1)"
    ),
]

print("🛡️ Constraint Violations — the database protects your data:")
print("=" * 65)

for label, sql in demos:
    try:
        conn = get_db()
        cursor = conn.cursor()
        cursor.execute(sql)
        conn.commit()
        conn.close()
        print(f"  ✅ {label}: Allowed (unexpected!)")
    except Exception as e:
        conn.close()
        error_msg = str(e).split('\n')[0]
        print(f"  ❌ {label}")
        print(f"     → {error_msg}")
        print()

print("💡 Constraints catch bugs at the database level — no bad data gets through,")
print("   even if your application code has a bug.")

🛡️ Constraint Violations — the database protects your data:
  ❌ UNIQUE constraint (duplicate email)
     → duplicate key value violates unique constraint "users_email_key"

  ❌ FOREIGN KEY constraint (non-existent user)
     → insert or update on table "posts" violates foreign key constraint "posts_user_id_fkey"



  ❌ NOT NULL constraint (missing content)
     → null value in column "content" of relation "posts" violates not-null constraint

  ❌ CHECK constraint (follow yourself)
     → new row for relation "follows" violates check constraint "follows_check"

💡 Constraints catch bugs at the database level — no bad data gets through,
   even if your application code has a bug.


## 🧹 Referential Actions: What Happens on Delete?

Foreign keys raise a question: when you delete the *parent* row, what happens to the *children*?

PostgreSQL lets you choose with `ON DELETE`:

| Action | Behavior | Use when... |
|--------|----------|-------------|
| `RESTRICT` (default) | Block the delete if children exist | The relationship is required and you want explicit cleanup |
| `CASCADE` | Auto-delete all children too | Children can't exist without the parent (e.g. comments on a deleted post) |
| `SET NULL` | Set child's FK to NULL | The relationship is *optional* (child survives without a parent) |

> ⚠️ **Don't default to `CASCADE` everywhere.** Use it only for true ownership relationships. For accidental-delete safety, prefer `RESTRICT`.


In [9]:
# Demo all three referential actions in temporary tables

execute("""
    DROP TABLE IF EXISTS demo_comments, demo_posts, demo_categories CASCADE;

    CREATE TABLE demo_categories (
        id SERIAL PRIMARY KEY,
        name TEXT
    );

    -- SET NULL: if category is deleted, post survives with category_id = NULL
    CREATE TABLE demo_posts (
        id SERIAL PRIMARY KEY,
        category_id INTEGER REFERENCES demo_categories(id) ON DELETE SET NULL,
        title TEXT
    );

    -- CASCADE: deleting a post deletes its comments
    CREATE TABLE demo_comments (
        id SERIAL PRIMARY KEY,
        post_id INTEGER REFERENCES demo_posts(id) ON DELETE CASCADE,
        body TEXT
    );

    INSERT INTO demo_categories (id, name) VALUES (1, 'tech');
    INSERT INTO demo_posts (id, category_id, title) VALUES (1, 1, 'Hello');
    INSERT INTO demo_comments (post_id, body) VALUES (1, 'Nice!'), (1, 'Agreed');
""")

print("Before deletion:")
print(f"  posts:    {query('SELECT * FROM demo_posts')}")
print(f"  comments: {query('SELECT * FROM demo_comments')}")

# Delete the post → CASCADE removes its comments automatically
execute("DELETE FROM demo_posts WHERE id = 1")
print("\nAfter DELETE post (CASCADE → comments auto-deleted):")
print(f"  posts:    {query('SELECT * FROM demo_posts')}")
print(f"  comments: {query('SELECT * FROM demo_comments')}")

# Re-create a post and demonstrate SET NULL on category
execute("INSERT INTO demo_posts (id, category_id, title) VALUES (2, 1, 'Another')")
execute("DELETE FROM demo_categories WHERE id = 1")
print("\nAfter DELETE category (SET NULL → post.category_id is now NULL):")
print(f"  posts: {query('SELECT * FROM demo_posts')}")

# Cleanup
execute("DROP TABLE IF EXISTS demo_comments, demo_posts, demo_categories CASCADE")
print("\n💡 In our main schema we left FKs as RESTRICT (default) for safety.")
print("   In production, choose per-relationship: comments→posts is a great fit for CASCADE.")


Before deletion:
  posts:    [RealDictRow({'id': 1, 'category_id': 1, 'title': 'Hello'})]
  comments: [RealDictRow({'id': 1, 'post_id': 1, 'body': 'Nice!'}), RealDictRow({'id': 2, 'post_id': 1, 'body': 'Agreed'})]

After DELETE post (CASCADE → comments auto-deleted):
  posts:    []
  comments: []



After DELETE category (SET NULL → post.category_id is now NULL):
  posts: [RealDictRow({'id': 2, 'category_id': None, 'title': 'Another'})]



💡 In our main schema we left FKs as RESTRICT (default) for safety.
   In production, choose per-relationship: comments→posts is a great fit for CASCADE.


## 📇 Indexes: Speeding Up Queries

An index is like the index in a book — instead of scanning every page, you jump straight to what you need.

Without an index, the database does a **sequential scan** (reads every single row).  
With an index, it does an **index scan** (jumps directly to matching rows).

**Rule of thumb:** Create indexes on columns you frequently filter or join on.

In [10]:
# Let's see what indexes exist on our tables
indexes = query("""
    SELECT
        tablename,
        indexname,
        indexdef
    FROM pg_indexes
    WHERE schemaname = 'public'
    ORDER BY tablename, indexname;
""")

print("📇 Indexes in our database:")
print("=" * 80)
current_table = None
for idx in indexes:
    if idx['tablename'] != current_table:
        current_table = idx['tablename']
        print(f"\n  📋 {current_table}:")
    # Extract just the column part from the index definition
    print(f"     {idx['indexname']}")

📇 Indexes in our database:

  📋 comments:
     comments_pkey
     idx_comments_post_id
     idx_comments_user_id

  📋 follows:
     follows_pkey
     idx_follows_following

  📋 likes:
     idx_likes_post_id
     idx_likes_user_id
     likes_pkey
     likes_user_id_post_id_key

  📋 post_tags:
     post_tags_pkey

  📋 posts:
     idx_posts_created_at
     idx_posts_user_id
     posts_pkey

  📋 posts_denormalized:
     posts_denormalized_pkey

  📋 tags:
     tags_name_key
     tags_pkey

  📋 users:
     users_email_key
     users_pkey
     users_username_key


In [11]:
# EXPLAIN shows how the database plans to execute a query
# This is how you check if an index is being used

print("🔍 Query plan: find posts by user_id (indexed column):")
print("=" * 60)
plan = query("EXPLAIN SELECT * FROM posts WHERE user_id = 1")
for row in plan:
    print(f"  {row['QUERY PLAN']}")

print()
print("🔍 Query plan: find posts by content (NOT indexed):")
print("=" * 60)
plan2 = query("EXPLAIN SELECT * FROM posts WHERE content LIKE '%pasta%'")
for row in plan2:
    print(f"  {row['QUERY PLAN']}")

print()
print("💡 Notice the first uses an Index Scan (fast) while the second")
print("   uses a Seq Scan (reads every row). That's the power of indexes!")

🔍 Query plan: find posts by user_id (indexed column):
  Seq Scan on posts  (cost=0.00..5.50 rows=5 width=79)
    Filter: (user_id = 1)

🔍 Query plan: find posts by content (NOT indexed):
  Seq Scan on posts  (cost=0.00..5.50 rows=34 width=79)
    Filter: (content ~~ '%pasta%'::text)

💡 Notice the first uses an Index Scan (fast) while the second
   uses a Seq Scan (reads every row). That's the power of indexes!


## 🧩 Composite Indexes & `EXPLAIN ANALYZE`

A **composite index** covers *multiple columns at once* — useful when a query filters/sorts on several fields together.

The crucial rule: a composite index on `(a, b)` helps queries that filter by **`a`** or by **`a` AND `b`**, but **not** queries that filter only by `b`. Think of a phone book sorted by *(last_name, first_name)*: you can find "Smith" quickly, and "Smith, John" instantly — but finding everyone named "John" still requires scanning the whole book. This is called the **left-prefix rule**.

`EXPLAIN ANALYZE` actually *runs* the query and reports the real plan + timing — much more informative than `EXPLAIN` alone.

> ⏱️ Exact milliseconds will vary in Docker / on a small dataset. Focus on:
> - the **plan shape** (Index Scan vs Seq Scan)
> - **rows estimated vs actual**
> - whether the right index is used


In [12]:
# A realistic feed query: posts by a specific user, newest first
sql = """
SELECT id, content, created_at
FROM posts
WHERE user_id = 1
ORDER BY created_at DESC
LIMIT 10
"""

# Our seed data is tiny (200 rows), so Postgres often prefers a Seq Scan
# regardless of indexes. To make the lesson clear we ask the planner to
# avoid Seq Scans for this demo only — in production, let the planner choose.
def explain(sql_text):
    conn = get_db()
    cursor = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    cursor.execute("SET LOCAL enable_seqscan = off")
    cursor.execute("SET LOCAL enable_bitmapscan = off")
    cursor.execute("EXPLAIN ANALYZE " + sql_text)
    rows = cursor.fetchall()
    conn.close()
    return rows

print("🔍 Plan WITHOUT composite index (forced to use an index → idx_posts_user_id):")
print("-" * 70)
for row in explain(sql):
    print(f"  {row['QUERY PLAN']}")

# Add a composite index that matches both the WHERE and ORDER BY
execute("CREATE INDEX IF NOT EXISTS idx_posts_user_created ON posts(user_id, created_at DESC)")

print("\n🔍 Plan WITH composite index on (user_id, created_at DESC):")
print("-" * 70)
for row in explain(sql):
    print(f"  {row['QUERY PLAN']}")

print("\n💡 Look for the difference:")
print("   - Without composite index: Index Scan on user_id, then a separate Sort step.")
print("   - With composite index: Index Scan returns rows ALREADY sorted by created_at,")
print("     so Postgres stops after the first 10 — no Sort node needed.")
print("   At small scale this saves microseconds. At millions of rows it's huge.")

# Demonstrate the left-prefix rule: a query filtering only by created_at
# CANNOT use the (user_id, created_at) index — it has to use a different one.
print("\n🧭 Left-prefix rule: filtering only by created_at canNOT use idx_posts_user_created.")
print("-" * 70)
for row in explain("SELECT * FROM posts WHERE created_at > NOW() - INTERVAL '7 days'"):
    print(f"  {row['QUERY PLAN']}")
print("   → Postgres falls back to the single-column idx_posts_created_at instead.")
print("   That\'s why composite-index column ORDER matters: put the most-filtered column first.")

# Cleanup the demo index so re-running the cell stays clean
execute("DROP INDEX IF EXISTS idx_posts_user_created")


🔍 Plan WITHOUT composite index (forced to use an index → idx_posts_user_id):
----------------------------------------------------------------------
  Limit  (cost=16.23..16.24 rows=5 width=75) (actual time=0.029..0.030 rows=5 loops=1)
    ->  Sort  (cost=16.23..16.24 rows=5 width=75) (actual time=0.028..0.028 rows=5 loops=1)
          Sort Key: created_at DESC
          Sort Method: quicksort  Memory: 25kB
          ->  Index Scan using idx_posts_user_id on posts  (cost=0.14..16.17 rows=5 width=75) (actual time=0.012..0.015 rows=5 loops=1)
                Index Cond: (user_id = 1)
  Planning Time: 0.261 ms
  Execution Time: 0.052 ms

🔍 Plan WITH composite index on (user_id, created_at DESC):
----------------------------------------------------------------------


  Limit  (cost=0.14..16.20 rows=5 width=75) (actual time=0.026..0.030 rows=5 loops=1)
    ->  Index Scan using idx_posts_user_created on posts  (cost=0.14..16.20 rows=5 width=75) (actual time=0.025..0.028 rows=5 loops=1)
          Index Cond: (user_id = 1)
  Planning Time: 0.325 ms
  Execution Time: 0.044 ms

💡 Look for the difference:
   - Without composite index: Index Scan on user_id, then a separate Sort step.
   - With composite index: Index Scan returns rows ALREADY sorted by created_at,
     so Postgres stops after the first 10 — no Sort node needed.
   At small scale this saves microseconds. At millions of rows it's huge.

🧭 Left-prefix rule: filtering only by created_at canNOT use idx_posts_user_created.
----------------------------------------------------------------------
  Index Scan using idx_posts_created_at on posts  (cost=0.15..16.30 rows=16 width=79) (actual time=0.026..0.034 rows=16 loops=1)
    Index Cond: (created_at > (now() - '7 days'::interval))
  Planning Time: 

---

> 🧭 **Pro tip — design from access patterns.** Before adding indexes, write down the queries you actually need. Our feed query below filters by `follower_id`, joins by `following_id`, sorts by `created_at`, and groups by `post_id`. Every index in `db/init.sql` exists to support one of those steps. **Indexes you don't need still cost write performance and disk space — don't add them speculatively.**


## 🧪 Practice: Building a Feed Query

Let's combine everything we learned to build a real feature: **a user's news feed**.

The feed shows recent posts from people the user follows, with like counts.

In [13]:
# Build a feed: recent posts from people user_1 follows

feed = query("""
    SELECT
        u.username,
        p.content,
        p.created_at,
        COUNT(l.id) AS like_count
    FROM posts p
    JOIN users u ON p.user_id = u.id
    -- Only posts from people user_1 follows
    JOIN follows f ON f.following_id = p.user_id AND f.follower_id = 1
    -- Count likes per post
    LEFT JOIN likes l ON l.post_id = p.id
    GROUP BY u.username, p.id, p.content, p.created_at
    ORDER BY p.created_at DESC
    LIMIT 10;
""")

print("📰 News Feed for @user_1:")
print("=" * 70)
for item in feed:
    likes = item['like_count']
    hearts = '❤️ ' * min(likes, 5) + (f'+{likes-5}' if likes > 5 else '')
    print(f"  @{item['username']}:")
    print(f"    {item['content'][:65]}")
    print(f"    {hearts} ({likes} likes)  •  {item['created_at']}")
    print()

print("💡 This single query uses 3 JOINs and a GROUP BY.")
print("   Relational databases make complex queries like this straightforward.")
print("   But at scale, this could be slow — we'll explore solutions in Notebook 2!")

📰 News Feed for @user_1:
  @user_46:
    Just shipped a new feature at work! Feeling productive 🚀
    ❤️ ❤️ ❤️ ❤️ ❤️ +3 (8 likes)  •  2026-04-14 18:51:35.879782

  @user_26:
    Morning run done! 5K in 25 minutes, new personal best 🏃
    ❤️ ❤️ ❤️ ❤️  (4 likes)  •  2026-04-09 10:24:38.272058

  @user_36:
    Working on a side project this weekend. Building a URL shortener!
    ❤️ ❤️ ❤️ ❤️ ❤️ +1 (6 likes)  •  2026-04-07 19:55:34.952614

  @user_19:
    Made homemade pasta for the first time — turned out amazing 🍝
    ❤️ ❤️ ❤️ ❤️ ❤️ +5 (10 likes)  •  2026-03-31 17:07:51.388078

  @user_36:
    Beautiful sunset from my balcony today 🌅
    ❤️ ❤️ ❤️ ❤️ ❤️ +3 (8 likes)  •  2026-03-30 21:37:22.748433

  @user_46:
    Currently reading "Designing Data-Intensive Applications" — highl
    ❤️ ❤️ ❤️ ❤️ ❤️ +2 (7 likes)  •  2026-03-30 14:56:45.567076

  @user_46:
    Morning run done! 5K in 25 minutes, new personal best 🏃
    ❤️ ❤️ ❤️ ❤️ ❤️  (5 likes)  •  2026-03-28 03:07:56.549326

  @user_19:
    M

## 📚 Summary

### Key Takeaways

1. **Entities become tables** — identify the "things" your app cares about
2. **Primary keys** uniquely identify rows — use system-generated IDs, not business data
3. **Foreign keys** create relationships — 1:N uses a single FK, N:M needs a junction table
4. **Constraints** protect data integrity at the database level
5. **Indexes** speed up queries on frequently filtered columns
6. **JOINs** combine data across tables — the core strength of relational databases

### Interview Tip

> When designing a schema in an interview, start by listing entities and their relationships.  
> Then add primary keys, foreign keys, and indexes that match your API endpoints.

### Next Up

In **Notebook 2**, we'll explore **denormalization** — when joins become too expensive and you need to trade storage for speed.